## OptiSpeech Training: HFC-Female (en-US)
This notebook allows you to train [OptiSpeech TTS](https://github.com/mush42/optispeech) on [HiFiCaptin en-US female dataset](https://ast-astrec.nict.go.jp/en/release/hi-fi-captain/)


## Plumming

In [1]:
#@markdown ### Google Colab Anti-Disconnect
#@markdown Avoid automatic disconnection. Still, it will disconnect after **6 to 12 hours**.

import IPython
js_code = '''
function ClickConnect(){
console.log("Working");
document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect,60000)
'''
display(IPython.display.Javascript(js_code))


#@markdown ### Check GPU type
#@markdown A higher capable GPU can lead to faster training speeds. By default, you will have a **Tesla T4**.
!nvidia-smi

<IPython.core.display.Javascript object>

Tue Jan 27 02:03:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Prepare environment

In [ ]:

import os

# 1. Install Python 3.11 and Dev tools globally
!apt-get update
!apt-get install python3.11 python3.11-dev python3.11-distutils -y

# 2. Register both versions in the 'update-alternatives' system
# We give Python 3.11 a higher priority (2) than Python 3.12 (1)
!update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.11 2
!update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.12 1

# 3. Fix Pip (changing Python versions often breaks the global pip link)
!curl https://bootstrap.pypa.io/get-pip.py -o get-pip.py
!python3 get-pip.py --force-reinstall

# 4. Verify the change
!python3 --version

if not os.path.isdir(os.path.join(os.getcwd(), "optispeech")):
    print("Cloning optispeech repository...")
    !git clone --branch eden-alighn --depth=1 https://github.com/mush42/optispeech


# Install OptiSpeech
%cd /content/optispeech
!pip install -e .

# Install some extra dependencies
!pip install transformers -U
!pip install onnxruntime soundfile numpy
!pip install torch torchvision torchaudio
!python3 -c "import torch; print(torch.__version__)"


## Preprocess Dataset

In [ ]:
%cd /content
!unzip -q /content/drive/MyDrive/Local_AI/hfc_female-en_us-dataset.zip
%cd /content/optispeech
!rm -rf data/hfc_female-en_us
!python3 -m optispeech.tools.preprocess_dataset \
    --format ljspeech \
    hfc_female-en_us \
    /content/hfc_female-en_us-dataset \
    /content/optispeech/data/hfc_female-en_us

# Backup preprocess data
%cd /content/optispeech/data
!zip -1 -r -q /content/drive/MyDrive/Local_AI/hfc_female-en_us_preprocessed_data.zip hfc_female-en_us


# Backup the Optispeech Preprocess Data

In [ ]:
%cd /content/optispeech/data
!zip -1 -r -q /content/drive/MyDrive/Local_AI/hfc_female-en_us_preprocessed_data.zip hfc_female-en_us


# Restore the Optispeech Preprocess Data

In [7]:
import os

%cd /content/optispeech/

if os.path.isdir(os.path.join(os.getcwd(), "data/hfc_female-en_us")):
    print("Remove old data...")
    !rm -rf data/hfc_female-en_us

if not os.path.isdir(os.path.join(os.getcwd(), "data")):
    !mkdir data

!unzip -q /content/drive/MyDrive/Local_AI/hfc_female-en_us_preprocessed_data.zip -d data

/content/optispeech


## Enable Tensorboard

In [ ]:
# Create log directory
!mkdir -p /content/drive/MyDrive/Local_AI/optispeech/logs

%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/Local_AI/optispeech/logs


# Patch The generic.py

In [10]:
import os

file_path = '/content/optispeech/optispeech/utils/generic.py'

# Read the original content of the file
with open(file_path, 'r') as f:
    content = f.read()

# Define the new and old function signatures/markers to find and replace
old_function_start_marker = "def save_figure_to_numpy(fig: plt.Figure) -> np.ndarray:"
old_function_end_marker = "    return data"

# The new implementation of the function
new_function_implementation = '''def save_figure_to_numpy(fig: plt.Figure) -> np.ndarray:
    """
    Converts a matplotlib figure to a numpy array.
    """
    import io
    from PIL import Image
    import numpy as np

    # Save the figure to a BytesIO object
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
    buf.seek(0)

    # Open the image with PIL and convert to a NumPy array
    img = Image.open(buf)
    data = np.array(img)
    return data'''

# Find the start and end of the old function
start_idx = content.find(old_function_start_marker)
# Look for the return statement specific to the old function structure
end_idx = content.find("data = np.fromstring(fig.canvas.tostring_rgb(), dtype=np.uint8, sep=\"\")", start_idx) # Find the problematic line

if start_idx != -1 and end_idx != -1:
    # Find the end of the logical block for the old function
    # This assumes the old function block ends shortly after the problematic line
    # To be precise, we need to find the end of the function definition
    # For simplicity and robustness, we will replace the entire function body

    # Assuming the problematic line is the last line before the function ends or another block starts.
    # Let's find the next 'def' or module-level code to get a boundary.
    # This is a heuristic, but often works for simple function replacements.
    next_def_or_class = content.find('\ndef ', end_idx)
    if next_def_or_class == -1:
        next_def_or_class = content.find('\nclass ', end_idx)
    if next_def_or_class == -1:
        next_def_or_class = len(content) # If no other definitions, go to end of file

    # Replace the old function definition and its body with the new one
    # Adjusting start_idx to include any leading indentation/newlines for the function definition
    # The new_function_implementation should include the 'def' line.
    modified_content = content[:start_idx] + new_function_implementation + content[next_def_or_class:]

    with open(file_path, 'w') as f:
        f.write(modified_content)
    print(f"Successfully patched {file_path}")
else:
    print(f"Could not find 'save_figure_to_numpy' function or its problematic line in {file_path}. Skipping patch.")

Successfully patched /content/optispeech/optispeech/utils/generic.py


## Start training

In [ ]:
%cd /content/optispeech
!python3 -m optispeech.train \
    experiment="hfc_female-en_us" \
    ++data.train_filelist_path="data/hfc_female-en_us/train.txt" \
    ++data.valid_filelist_path="data/hfc_female-en_us/val.txt" \
    ++data.batch_size=64 \
    ++data.num_workers=2 \
    ++model.train_args.evaluate_utmos=false \
    ++model.train_args.evaluate_pesq=false \
    ++callbacks.model_checkpoint.every_n_epochs=5 \
    ++callbacks.model_checkpoint.save_last=true \
    ++paths.log_dir="/content/drive/MyDrive/Local_AI/optispeech/logs"


/content/optispeech
[2026-01-27 02:25:47,123][optispeech.utils.generic][INFO] - Enforcing tags! <cfg.extras.enforce_tags=True>
[2026-01-27 02:25:47,137][optispeech.utils.generic][INFO] - Printing config tree with Rich! <cfg.extras.print_config=True>
CONFIG
├── data
│   └── _target_: optispeech.dataset.TextWavDataModule                          
│       name: hi-fi_en-US_female                                                
│       train_filelist_path: data/hfc_female-en_us/train.txt                    
│       valid_filelist_path: data/hfc_female-en_us/val.txt                      
│       batch_size: 64                                                          
│       num_workers: 2                                                          
│       pin_memory: true                                                        
│       data_statistics:                                                        
│         pitch_min: 59.842995                                                  
│    